# étoiles symbiotiques 
$\rightarrow$ **analyse de l'étoile AG Peg**

## la cible

AG Peg est une étoile double 'symbiotique' - les différences entre Étoiles Symbiotiques et Variables Cataclysmiques sont :

La nature du "Donneur" (l'étoile compagne)
- Symbiotiques (ex: AG Peg) : La compagne est une géante rouge (type spectral M). Le système est très large (périodes orbitales de plusieurs centaines de jours, 818 jours pour AG Peg).
- Cataclysmiques (CV) : La compagne est généralement une étoile naine de la série principale (type K ou M). Le système est très serré (périodes orbitales de quelques heures).

Le mode de transfert de masse
- Symbiotiques : Le transfert se fait principalement par la capture du vent stellaire massif de la géante rouge. Le gaz baigne tout le système, créant une nébulosité étendue.
- Cataclysmiques : Le transfert se fait par débordement du lobe de Roche. La matière tombe directement vers la naine blanche en formant un disque d'accrétion très structuré et brillant.

Signature Spectrale
- Symbiotiques : Un "spectre combiné" présentant à la fois des bandes moléculaires froides (TiO) et des raies d'émission de très haute ionisation (He II).
- Cataclysmiques : Le spectre est dominé par le disque d'accrétion (raies d'émission de l'hydrogène larges et souvent à double pic à cause de la rotation). On voit rarement la signature de l'étoile compagne car elle est trop faible par rapport au disque.

Phénomènes Éruptifs
- Symbiotiques : Produisent des novae lentes (comme l'éruption séculaire d'AG Peg) qui durent des décennies. L'hydrogène brûle de manière stable à la surface de la naine blanche.
- Cataclysmiques : Produisent des novae classiques, des novae naines (sursauts du disque) ou des novae récurrentes (comme T CrB). Les éruptions sont beaucoup plus brutales et brèves (quelques jours à quelques mois).

## les données 
|||
|---|---|
|OBJECT|	AG Peg|
|EXPTIME2|	5 x 300 s|
|DATE-OBS|	 2025-08-24T21:29:45|
|BSS_SITE|	RENNES|
|BSS_INST|	SW400/FD5 + Dados200 + 25mic + ATIK420M|
|||



La phase $\phi$ orbitale ($\phi = \text{fraction de } \frac{(T_{obs} - T_0)}{P}$) : 
- JD-OBS = 2460912.396 ($T_0 = 2446812$, $P = 818,2$ jours)
- Jours écoulés : $14100,396$
- Cycles : $17,233$ -> phase orbitale : 0,23 -> la géante rouge commence à s'éloigner de la ligne de visée par rapport à la naine blanche.
- Les deux composantes sont bien séparées -> raies d'émission nettes sans absorption excessive par le vent dense de la géante.

Résolution spectrale $R$ est de 646 : À $H\alpha$ ($6563$ Å), la largeur instrumentale minimale est de :$$\Delta\lambda = \frac{6563}{646} \approx 10,1 \text{ Å}$$

Échantillonnage (pas spectral) : CDELT1 = 0.875 Å/pixel-> 11,5 pixels couvre un élément de résolution ($10,1 / 0,875$) -> échantillonnage suffisant pour calculer les EW

correction héliocentrique non appliquée.

**Attention** : le continuum est complexe à cause de la contribution de la géante rouge (bandes moléculaires TiO dans le rouge) -> le continuum ne peut pas être retiré par un fit global



## création du dashboard
- lancer la cellule suivante
- souris sur '**Colormap**', bouton droit, menu "**create new view for cell output**"
- déplacer le nouvel onglet créé '**Output View**' pour le garder visible pendant que vous naviguez et exécutez les cellules de code

En cas de souci d'affichage $\rightarrow$  '**CTRL-R**'


In [1]:
%matplotlib widget
from spectro_dashboard import SpectroDashboard

db = SpectroDashboard()
db.show()


# affichage

In [2]:
# imports
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
from specutils import Spectrum
from astropy.modeling.models import Gaussian1D, Voigt1D, Lorentz1D, Const1D
from astropy.modeling.fitting import LevMarLSQFitter # L'outil de fit correct

from specutils.fitting import fit_generic_continuum
from specutils.manipulation import extract_region

from astropy.modeling.models import Polynomial1D, Chebyshev1D, Legendre1D, Linear1D
from specutils import SpectralRegion
from astropy.stats import mad_std
from specutils.analysis import snr, snr_derived
from specutils.spectra import SpectralRegion

from astropy.constants import c

from astropy import units as u
from astropy.nddata import CCDData

import pandas as pd


from PIL import Image
db.show_image(Image.open('data/plouis/agpeg.jpg').convert('L'), 'brut')


# on affiche un brut, le spectre calibré et le header FITS

img = CCDData.read('data/plouis/AGPeg-300s-5.fits', unit=u.Unit('adu'))
db.show_image(img, f"brut de {img.meta['EXPTIME']}s")

#db.clear_spectra()
spec_name = 'data/plouis/_agpeg_20250824_896.fits'
spec = Spectrum.read(spec_name)
db.show_spectrum(spec.wavelength, spec.flux, label=spec_name)

# on montre le header
#header = spec.meta['header']
#df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])
#display(df.style.hide(axis='index'))   


INFO: affichage de l'image brut : bin=1, shape=(1500, 2154), min=0, avg=80.1, max=230, stddev=15.4
INFO: affichage de l'image brut de 300s : bin=1, shape=(1219, 1619), min=1052, avg=1267.7, max=65535, stddev=147.2


INFO: affichage du spectre 'data/plouis/_agpeg_20250824_896.fits' : 2849 pts, X:[4251.2:6743.5]


# on calcule le SNR

In [3]:
#db.clear_spectra()

region_continuum = SpectralRegion(5500.0 * u.AA, 5800.0 * u.AA)

# Calcul des erreurs systématiques (DADOS 200)
fwhm_neon_pix = 5   #px
delta_lambda = np.mean(np.diff(spec.spectral_axis)) # Dispersion (A/pix)

err_fwhm_inst_angstrom = (fwhm_neon_pix * delta_lambda) / 10
print(f"Erreur systématique (1/10 néon): {err_fwhm_inst_angstrom:.2f}")

sub_spectrum = extract_region(spec, region_continuum)
snr_val = np.median(sub_spectrum.data) / mad_std(sub_spectrum.data)
print(f"SNR : {snr_val:.0f}")


Erreur systématique (1/10 néon): 0.44 Angstrom
SNR : 12


# on calcule les EW de Hbeta et HeII

In [4]:
import numpy as np
from specutils.analysis import equivalent_width
from specutils import SpectralRegion
from specutils.manipulation import extract_region
import astropy.units as u

# calcule la largeur équivalente en prenant un continuum de 1.0 proches des raies
# indispensable ici car le spectre de la géante est rempli de TiO qui ne brouille le contiuum ...
def get_ew(spectrum, snr_value, line_start, line_end, cont_start, cont_end):
    line_reg = SpectralRegion(line_start*u.AA, line_end*u.AA)
    cont_reg = SpectralRegion(cont_start*u.AA, cont_end*u.AA)
    
    # On extrait la zone du continuum et on prend la moyenne
    cont_sub = extract_region(spectrum, cont_reg)
    mean_cont = np.mean(cont_sub.flux)
    
    # On divise tout le spectre par la valeur moyenne du continuum
    spec_norm = spectrum / mean_cont

    # on vérifie que c'est bon visuellement
    #plt.figure(figsize=(10, 4))
    #plt.plot(spec_norm.spectral_axis, spec_norm.flux)
    #plt.axhline(1.0, color='r', linestyle='--') # La ligne du continuum
    #plt.xlim(line_start - 20, line_end + 20) # On zoome sur la raie
    #plt.ylim(0, np.max(spectrum.flux))
    #plt.show()

    # on récupère l'EW
    ew_value = equivalent_width(spec_norm, regions=line_reg)

    # on calcule l'incertitude
    delta_lambda = np.mean(np.diff(spectrum.spectral_axis))       #  pas spectral (dispersion)
    err_ew_fit = (1.5 / snr_value) * np.sqrt(delta_lambda * np.abs(ew_value))     # formule de Vollmann & Bernhard (Cayrel généralisée) :

    return (abs(ew_value), err_ew_fit)

ew_hbeta, sigma_ew_hbeta = get_ew(spec, snr_val, 4845, 4875, 4840, 4845)
ew_heii, sigma_ew_heii  = get_ew(spec, snr_val, 4670, 4705, 4670, 4675)

print(f"EW H-beta : {ew_hbeta.value:.2f} +/- {np.sqrt(sigma_ew_hbeta**2 + err_fwhm_inst_angstrom**2):.2f}")
print(f"EW He-II  : {ew_heii.value:.2f} +/- {np.sqrt(sigma_ew_heii**2 + err_fwhm_inst_angstrom**2):.2f}")


EW H-beta : 24.70 +/- 0.73 Angstrom
EW He-II  : 30.20 +/- 0.78 Angstrom


# on calcule la température de la naine blanche

Les modèles théoriques (Iijima, 1981, Mukai, 1988) et Skopal 2005, A&A 440) établissent une relation directe entre le ratio  ew_heii / ew_hbeta et la température de la source stellaire ($T_{wd}$) :


$$T_{eff} = \left( 45,70 \times \sqrt{R} + 23,20 \right) \times 10^3 \text{ K}$$




In [5]:
ratio = ew_heii / ew_hbeta
print(f"{ratio=:.2f}")

# propagation sur ratio = ew_heii / ew_hbeta
err_ratio = ratio * np.sqrt((sigma_ew_heii/ew_heii)**2 + (sigma_ew_hbeta/ew_hbeta)**2)

#t_eff = (19.38 * np.sqrt(2.22 * ratio + 1.23) + 5.1) * 1000
#t_eff = np.sqrt(ratio/0.0034)*1000
t_naine_blanche = (45.7 * np.sqrt(ratio) + 23.2) * 1000     # Skopal 2005, A&A 440

# propagation sur T = (45.7 * sqrt(ratio) + 23.2) * 1000
dT_dratio = 45.7 * 1000 / (2 * np.sqrt(ratio))
err_T = dT_dratio * err_ratio
print(f"Température naine blanche = {t_naine_blanche:.0f} +/- {err_T:.0f} K")


ratio=1.22
Température naine blanche = 73740 +/- 806 K


# on calcule le taux d'accrétion

In [6]:
# --- Constantes Physiques pour AG Peg ---
dist_pc = 800
dist_cm = dist_pc * 3.086e18
r_wd_cm = 4.17e9      # ~0.06 Rayon Solaire
m_wd_g = 1.0e33       # ~0.5 Masse Solaire
g_const = 6.674e-8    # Constante gravitationnelle (CGS)
m_sun_year = 1.989e33 / (365.25 * 24 * 3600) # Conversion g/s en M_sol/an
v_mag = 8.8     # magnitude an aout 2025 - source = AAVSO

# Estimation du flux du continuum à 4861A basé sur la magnitude V
# (Formule simplifiée : f_lambda = 10**(-0.4 * V - 8.45))
flux_cont = 10**(-0.4 * v_mag - 8.44) 

# Flux à 4861 Å depuis la magnitude V, avec correction de couleur B-V
b_v = 1.2  # typique pour AG Peg (géante rouge + naine blanche)
flux_cont_hbeta = flux_cont * 10**(-0.4 * 0.6 * b_v)  # correction approximative

# Flux absolu de H-beta (erg/s/cm2)
flux_hbeta = ew_hbeta * flux_cont

# Luminosité de H-beta (erg/s)
lum_hbeta = 4 * np.pi * (dist_cm**2) * flux_hbeta

# Luminosité d'accrétion totale (approx.  L_acc ~ 100 * L_hb)
lum_acc = 100 * lum_hbeta

# M_dot en g/s puis en Masse Solaire / an
m_dot_gs = (lum_acc * r_wd_cm) / (g_const * m_wd_g)
m_dot_year = m_dot_gs / m_sun_year

print(f"Taux d'accrétion : {m_dot_year.value:.2e} masses_solaire / an")


Taux d'accrétion : 2.06e-07 masses_solaire / an


-> C'est un taux d'accrétion **modéré**, typique d'une phase de calme après son sursaut de 2015.

En comparaison : Lors de son éruption en 2015, ce taux était 10 à 50 fois plus élevé.